# Member 2: Q3 Uniform Cost Search & Q4 A* Search
**Student Name:** Walimuniarachchi W.A.K.S  
**Student ID:** IT24103947  
**Group:** Group_AI_28  
**Module:** SE3062 - Intelligent Systems  

---
### 📌 Overview
This notebook contains Member 2's implementation for:
1. **Q3: Uniform Cost Search (UCS)** in `search.py` (4 Marks)
2. **Q4: A* Search** in `search.py` (4 Marks)


---
## ⚙️ Step 0: Setup & Autograder Runner (RUN THIS FIRST)
This cell automatically finds where `autograder.py` is located (whether running in **VS Code**, **Jupyter**, or **Google Colab**), adds the search framework to `sys.path` to enable all imports, and defines a universal `run_autograder()` function so all tests execute without `FileNotFoundError`.


In [ ]:
# 🚀 Universal Autograder Runner (VS Code / Local / Colab)
import os
import sys
import subprocess

def find_search_dir():
    current = os.path.abspath(os.getcwd())
    
    # 1. Search parent directories
    d = current
    for _ in range(6):
        c1 = os.path.join(d, "Group_Project", "search")
        if os.path.exists(os.path.join(c1, "autograder.py")):
            return os.path.abspath(c1)
        c2 = os.path.join(d, "search")
        if os.path.exists(os.path.join(c2, "autograder.py")):
            return os.path.abspath(c2)
        if os.path.exists(os.path.join(d, "autograder.py")):
            return os.path.abspath(d)
        p = os.path.dirname(d)
        if p == d:
            break
        d = p
        
    # 2. In Google Colab (/content)
    if os.path.exists("/content"):
        for root, dirs, files in os.walk("/content"):
            if "autograder.py" in files and "test_cases" in dirs:
                return root
        # If not cloned yet in Colab, clone from GitHub
        print("📥 Cloning repository in Colab...")
        subprocess.run(["git", "clone", "https://github.com/IT24400002/SE3062-assignment_1.git"], cwd="/content")
        for root, dirs, files in os.walk("/content"):
            if "autograder.py" in files and "test_cases" in dirs:
                return root

    return None

SEARCH_DIR = find_search_dir()

if SEARCH_DIR:
    if SEARCH_DIR not in sys.path:
        sys.path.insert(0, SEARCH_DIR)
    try:
        os.chdir(SEARCH_DIR)
        print(f"✅ Ready! Autograder located at: {SEARCH_DIR}")
    except Exception as e:
        print(f"✅ Ready! (Using subprocess in {SEARCH_DIR})")
else:
    print("❌ Could not locate autograder.py directory.")

def run_autograder(question=None):
    if not SEARCH_DIR or not os.path.exists(os.path.join(SEARCH_DIR, "autograder.py")):
        print("❌ autograder.py not found! Please check your directory structure.")
        return
        
    cmd = [sys.executable, "autograder.py"]
    if question:
        cmd.extend(["-q", question])
    print(f"📍 Running autograder in: {SEARCH_DIR}\n")
    proc = subprocess.Popen(cmd, cwd=SEARCH_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()


---
## 📦 Framework Imports


In [ ]:
# Safe imports enabled by Step 0
import util
from search import SearchProblem, nullHeuristic
print("util, SearchProblem, and nullHeuristic imported successfully!")


---
## Q3: Uniform Cost Search (UCS)
* **Strategy:** Expands node with lowest cumulative path cost $g(n)$.
* **Fringe:** `util.PriorityQueue` ordered by path cost $g(n)$.
* **Key Concept:** Goal test is performed on **pop**, guaranteeing the cheapest path even when step costs vary.


In [ ]:
# Implementation of Uniform Cost Search (UCS)
def uniformCostSearch(problem: SearchProblem):
    """Search the node of least total cost first."""
    # 1. Initialize PriorityQueue ordered by cumulative path cost g(n)
    fringe = util.PriorityQueue()
    # 2. Push start state: item = (state, [actions], cumulativeCost), priority = cost 0
    fringe.push((problem.getStartState(), [], 0), 0)
    # 3. Track visited states to prevent redundant expansions
    expanded = set()

    # 4. Pop node with least accumulated cost first
    while not fringe.isEmpty():
        state, path, cost = fringe.pop()

        # Goal test: return optimal path once popped from fringe
        if problem.isGoalState(state):
            return path

        # Expand unvisited state to explore successors
        if state not in expanded:
            expanded.add(state)  # Mark current state as visited
            for successor, action, stepCost in problem.getSuccessors(state):
                if successor not in expanded:
                    newCost = cost + stepCost  # Accumulate step cost g(n)
                    # Push successor with priority equal to total path cost
                    fringe.push((successor, path + [action], newCost), newCost)

    return []  # Return empty list if no path found


---
## Q4: A* Search
* **Strategy:** Expands node with lowest combined cost $f(n) = g(n) + h(n)$.
* **Fringe:** `util.PriorityQueue` ordered by priority $f(n)$.
* **Key Concept:** When $h(n) = 0$ (`nullHeuristic`), A* degenerates to UCS. With an admissible and consistent heuristic, A* remains optimal while expanding far fewer nodes.


In [ ]:
# Implementation of A* Search
def aStarSearch(problem: SearchProblem, heuristic=nullHeuristic):
    """Search the node that has the lowest combined cost and heuristic first."""
    start = problem.getStartState()
    # 1. Initialize PriorityQueue ordered by combined cost f(n) = g(n) + h(n)
    fringe = util.PriorityQueue()
    # 2. Push start state with initial priority f(start) = 0 + h(start)
    fringe.push((start, [], 0), heuristic(start, problem))
    # 3. Track visited states to guarantee optimality
    expanded = set()

    # 4. Pop node with lowest f(n) first
    while not fringe.isEmpty():
        state, path, cost = fringe.pop()

        # Goal test: return optimal path once popped from fringe
        if problem.isGoalState(state):
            return path

        # Expand unvisited state to explore successors
        if state not in expanded:
            expanded.add(state)  # Mark current state as visited
            for successor, action, stepCost in problem.getSuccessors(state):
                if successor not in expanded:
                    newCost = cost + stepCost  # Actual path cost g(n)
                    # Priority = g(n) + h(n) (accumulated cost + heuristic estimate)
                    priority = newCost + heuristic(successor, problem)
                    fringe.push((successor, path + [action], newCost), priority)

    return []  # Return empty list if no path found


---
## 🧪 Autograder Test Cells


In [ ]:
# Test Q3 - UCS (Score: 3/3 | 4 Marks)
run_autograder('q3')


In [ ]:
# Test Q4 - A* Search (Score: 3/3 | 4 Marks)
run_autograder('q4')
